# Spatial PCD Data Release

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

import anndata as ad
import scanpy as sc
import cupy as cp
from sklearn.neighbors import KNeighborsClassifier

import time
import rapids_singlecell as rsc

import rmm
from rmm.allocators.cupy import rmm_cupy_allocator

rmm.reinitialize(
    managed_memory=True, 
    pool_allocator=True,  
    devices=0  
)
cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:
sc.settings.set_figure_params(
    dpi=80,
    facecolor='white',
    frameon=False,
    fontsize=14,
    dpi_save=200,
    figsize=(6,6),
    format='png'
)

In [ ]:
adata = sc.read_h5ad('../data/raw-all-tissdiss.h5ad')

In [ ]:
adata = adata[adata.X.sum(axis=1) > 0]

mt_mask   = adata.var_names.str.startswith('MT-')
ribo_mask = adata.var_names.str.startswith(('RPS', 'RPL'))
ig_mask   = adata.var_names.str.startswith(('IGK', 'IGL', 'IGH'))

adata = adata[:, ~(mt_mask | ribo_mask | ig_mask)]
adata

# Rapids Workflow

In [ ]:
adata = adata.copy()

In [ ]:
rsc.get.anndata_to_GPU(adata)

In [ ]:
rsc.pp.normalize_total(adata)
rsc.pp.log1p(adata)

## Quality Check

In [ ]:
metrics = ['log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_mt']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, metrics):
  vals = adata.obs[col].values
  ax.hist(vals, bins=40, density=True, alpha=0.4, color='steelblue')
  xgrid = np.linspace(vals.min(), np.percentile(vals, 99), 500)
  ax.plot(xgrid, gaussian_kde(vals)(xgrid), color='steelblue', lw=2)
  ax.set_xlabel(col)
  ax.set_ylabel('density')

fig.tight_layout()

In [ ]:
adata = adata[
      (adata.obs['log1p_n_genes_by_counts'] > 5) &
      (adata.obs['log1p_n_genes_by_counts'] < 9.8) &
      (adata.obs['log1p_total_counts'] > 5.5) &
      (adata.obs['log1p_total_counts'] < 11) &
      (adata.obs['pct_counts_mt'] < 6)
  ]

In [ ]:
metrics = ['log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_mt']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, metrics):
  vals = adata.obs[col].values
  ax.hist(vals, bins=40, density=True, alpha=0.4, color='steelblue')
  xgrid = np.linspace(vals.min(), np.percentile(vals, 99), 500)
  ax.plot(xgrid, gaussian_kde(vals)(xgrid), color='steelblue', lw=2)
  ax.set_xlabel(col)
  ax.set_ylabel('density')

fig.tight_layout()

## Processing

In [ ]:
rsc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.25)
rsc.pp.scale(adata, max_value=10, zero_center=False)
rsc.tl.pca(adata, svd_solver='arpack')
adata.obsm['X_pca_temp'] = adata.obsm['X_pca']

In [ ]:
adata

In [ ]:
rmm.reinitialize(managed_memory=False, pool_allocator=True, devices=0)
cp.cuda.set_allocator(rmm_cupy_allocator)
rsc.pp.harmony_integrate(adata, key=["source_file", "TIS_ID_base"], flavor="harmony1")

In [ ]:
rsc.pp.neighbors(adata, n_neighbors=15, n_pcs=40, use_rep="X_pca_harmony", key_added="harmony")
rsc.tl.umap(adata, neighbors_key="harmony", key_added="X_umap_harmony", min_dist=0.001, spread=5)
rsc.tl.tsne(adata, n_pcs=40, early_exaggeration=12, learning_rate=200, use_rep="X_pca_harmony")
rsc.tl.leiden(adata, resolution=1.5, neighbors_key="harmony", key_added="leiden_harmony")

## KNN Label Transfer

In [ ]:
# KNN label transfer for unmatched (new) cells in shared Harmony PCA space
n_nbh = 100

adata.obs['knn_labels'] = adata.obs['old_labels']
X = adata.obsm['X_umap_harmony']
labels = adata.obs['knn_labels'].copy()
known  = labels.notna()
unknown = labels.isna()

knn = KNeighborsClassifier(n_neighbors=n_nbh, weights='distance', n_jobs=-1)
knn.fit(X[known], labels[known])

adata.obs.loc[unknown, 'knn_labels'] = knn.predict(X[unknown])

proba = knn.predict_proba(X[unknown])
adata.obs['knn_confidence'] = np.nan
adata.obs.loc[unknown, 'knn_confidence'] = proba.max(axis=1)

print("KNN predictions:")
print(adata.obs.loc[unknown, 'knn_labels'].value_counts())
print(adata.obs.loc[unknown, 'knn_confidence'].describe())

In [ ]:
sc.pl.embedding(adata, basis="X_umap_harmony", color="knn_labels", size=2)

In [ ]:
labels = pd.read_csv(
    "../data/Tiss_Diss_rank_leiden_harmony_OCT.xlsx - Unified_Manual_Labels.csv",
    dtype={"Cluster": str},
).set_index("Cluster")

adata.obs["annotation"] = adata.obs["leiden_harmony"].map(labels["Unified label"]).astype("category")
adata.obs["annotation_short"] = adata.obs["leiden_harmony"].map(labels["Short"]).astype("category")

In [ ]:
adata.write('../data/tissdiss_object.h5ad')

## Leiden resolution sweep (0.1 - 1.5)

In [ ]:
adata = sc.read_h5ad('../data/tissdiss_object.h5ad')

In [ ]:
RESOLUTIONS = [round(0.1 * i, 2) for i in range(1, 15)]  # 0.1, 0.2, …, 1.4
sweep_keys = [f'leiden_res{res:.2f}' for res in RESOLUTIONS]

for res, key in zip(RESOLUTIONS, sweep_keys):
    rsc.tl.leiden(adata, resolution=res, neighbors_key="harmony", key_added=key)

    rank_key = f'rank_{key}'
    rsc.tl.rank_genes_groups(adata, groupby=key, method="wilcoxon", use_raw=False, key_added=rank_key)

    deg = sc.get.rank_genes_groups_df(adata, group=None, key=rank_key)
    deg.to_csv(f'../data/tissdiss_deg_{key}_wilcoxon.csv', index=False)

In [ ]:
sc.pl.embedding(adata, color=sweep_keys, basis='X_umap_harmony', wspace=0.05, ncols=3,
                legend_loc='on data', legend_fontoutline=2, size=6)

In [ ]:
from pathlib import Path

highlight_ncols = 4
FIGURES_DIR = Path('../figures/cluster_highlights')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

for res, key in zip(RESOLUTIONS, sweep_keys):
    clusters = adata.obs[key].cat.categories.tolist()
    n_panels = len(clusters) + 1
    highlight_nrows = (n_panels + highlight_ncols - 1) // highlight_ncols
    fig, axes = plt.subplots(highlight_nrows, highlight_ncols,
                             figsize=(5 * highlight_ncols, 5 * highlight_nrows))
    axes = axes.flatten()

    sc.pl.embedding(adata, basis='X_umap_harmony', color=key, ax=axes[0], size=6, show=False,
                    legend_loc='on data', legend_fontoutline=2, title=f'res {res:.2f}: all clusters')

    for i, cluster in enumerate(clusters):
        sc.pl.embedding(adata, basis='X_umap_harmony', color=key, groups=[cluster], ax=axes[i + 1],
                        size=6, show=False, title=f'res {res:.2f}: cluster {cluster}',
                        na_color='#d3d3d3', legend_loc=None)

    for ax in axes[n_panels:]:
        ax.set_visible(False)

    fig.tight_layout()
    for extension in ('png', 'pdf'):
        fig.savefig(FIGURES_DIR / f'tissdiss_umap_cluster_highlights_res{res:.2f}.{extension}', bbox_inches='tight')
    plt.show()
    plt.close(fig)

## Visualization 

In [ ]:
sc.pl.embedding(adata, basis="X_umap_harmony", color=["TIS_ID_base", "source_file"], wspace = 0.8, size=2)

In [ ]:
adata = sc.read_h5ad('../data/tissdiss_object.h5ad')

In [ ]:
sc.pl.embedding(adata, basis="X_umap_harmony", color=["annotation", "leiden_harmony"], size=2, legend_loc="on data", legend_fontoutline=2)

## Diffusion Maps

In [ ]:
adata = sc.read_h5ad('../data/tissdiss_object.h5ad')

In [ ]:
rsc.tl.diffmap(adata, neighbors_key="harmony")
# Due to an issue with plotting in Scanpy, we need to shift the first component of the diffusion map by removing the first column:
adata.obsm["X_diffmap"] = adata.obsm["X_diffmap"][:, 1:]

In [ ]:
sc.pl.diffmap(adata, color="knn_labels")

## DEG Analysis

In [ ]:
rsc.tl.rank_genes_groups(
    adata, groupby="knn_labels", method="t-test", key_added="ttest_knn_labels", use_raw=False
)

result = adata.uns['ttest_knn_labels']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:5]: result[key][group]
    for group in groups for key in ['names', 'logfoldchanges', 'pvals_adj', 'scores']})

threshold = 0.05
significant_mask = pd.concat(
    [df[group + '_pvals'] < threshold for group in groups], axis=1).any(axis=1)
significant_df = df[significant_mask]
significant_df.head()
significant_df.to_csv('../data/rank_genes-ttest_knn_labels.csv')

In [ ]:
rsc.tl.rank_genes_groups(
    adata, groupby="leiden_harmony", method="t-test", key_added="ttest_leiden_harmony", use_raw=False
)

result = adata.uns['ttest_leiden_harmony']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:5]: result[key][group]
    for group in groups for key in ['names', 'logfoldchanges', 'pvals_adj', 'scores']})

threshold = 0.05
significant_mask = pd.concat(
    [df[group + '_pvals'] < threshold for group in groups], axis=1).any(axis=1)
significant_df = df[significant_mask]
significant_df.head()
significant_df.to_csv('../data/rank_genes-ttest_leiden_harmony.csv')

In [ ]:
adata.write('../data/tissdiss_object_with_ranks.h5ad')

## Group fractions

In [ ]:
pd.crosstab(      
      adata.obs['knn_labels'],
      adata.obs['leiden_harmony'],
      normalize='columns'  # fraction of each knn cluster
  ).style.background_gradient(cmap='Blues', axis=1)

In [ ]:
pd.crosstab(      
      adata.obs['leiden_harmony'],
      adata.obs['knn_labels'],
      normalize='index'  # fraction of each leiden cluster
  ).style.background_gradient(cmap='Reds', axis=1)

## Cluster views

In [ ]:
adata = sc.read_h5ad('../data/tissdiss_object_with_ranks.h5ad')

In [ ]:
adata

In [ ]:
sc.pl.embedding(adata, basis='X_umap_harmony', color=['disease_state', 'tissue_type', 'subject_id'], size=2, wspace = 0.8)

In [ ]:
sc.pl.embedding(adata, basis='X_umap_harmony', color=['annotation'], size=2)